## Build Your Own CNN

You've now gone from knowing nothing about ML to classification, MLPs, tabular data, trees, image recognition, and CNNs. That's a lot of ground. This project is where you pull it together.

Pick a dataset, build a CNN, train it, evaluate it, and iterate. The goal isn't to get great accuracy. It's to go through the full pipeline independently and notice what's familiar. Because most of it will be. Loading data, splitting it, batching, building a model, writing a training loop, plotting loss curves, diagnosing overfitting: you've done all of this before. A CNN is just another neural network with some domain-specific layers in the middle.

Don't get stuck on CNN-specific details like padding or stride. Those are mechanical, you set them once and move on. The skills that matter are the ones you've been building across every lesson: understanding your data before modeling it, recognizing overfitting from training curves, knowing when a model is too big or too small, using augmentation as regularization, reading a confusion matrix to understand failure patterns. These repeat everywhere in ML, whether you're working with images, text, audio, or tabular data.

Pick a dataset below, then follow the pipeline guide. Work interactively, try things yourself first, and prompt when you're stuck.

## Pick a Dataset

### Available via torchvision

These datasets can be downloaded automatically using `torchvision.datasets`. Pass `download=True` and a local path like `../../data/` to cache them.

**CIFAR-10** (60,000 32x32 color images, 10 classes)

Classic benchmark. Zero preprocessing needed, works directly with `torchvision.datasets.CIFAR10`. Good if you want to focus purely on architecture and training without data wrangling. CNNs hit 90%+ which is satisfying.

**Oxford Pets** (7,393 images of 37 pet breeds)

Real photographs with variable sizes. Forces realistic preprocessing: resize decisions, augmentation, annotation file parsing, custom Dataset class. Fine-grained breed classification is hard (expect 40-70% on a subset), but the pipeline practice is representative of real work.

**Flowers102** (8,189 images of 102 flower species)

Beautiful high-res images, but 102 classes from scratch is brutal. Great for transfer learning comparison later. Labels are in .mat format (needs scipy).

**Food101** (101,000 images of 101 food categories)

Large and well-balanced (1,000 per class). Practical application, but slow training and 101 classes is hard. Consider using a subset of 10-20 categories.

**CIFAR-100** (60,000 32x32 images, 100 fine-grained classes)

Same format as CIFAR-10 but much harder. Has 20 "superclasses" if you want an easier grouping. Tiny images make fine-grained classification tough.

**Fashion-MNIST** (70,000 28x28 grayscale images of clothing, 10 classes)

Drop-in MNIST replacement. Familiar format, but similar to what you've already done. Good if you want a quick warmup.

### Real-world datasets (download from Kaggle)

These are problems where CNNs are actually deployed in production.

**Chest X-Ray Pneumonia** (5,856 chest X-rays, Normal vs Pneumonia)
[Kaggle download](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)

Binary classification with clear visual differences. Already split into train/val/test folders, so you can focus on the model. Grayscale images, so your first conv layer takes 1 channel instead of 3. Satisfying accuracy (90%+), and this is genuinely what hospital screening systems are built on.

**NEU Steel Surface Defects** (1,800 grayscale images, 6 defect types: cracks, inclusions, pitting, etc.)
[Kaggle download](https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database)

Only 300 images per class. This is tiny, so your model will overfit hard unless you design augmentation carefully and keep the architecture small. That constraint is the whole point. It's also exactly what factory vision systems do: catch surface defects on a production line.

**PlantVillage** (54,000 leaf images, 38 classes covering healthy vs diseased across crops)
[Kaggle download](https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset)

Clean lab-shot images on uniform backgrounds. A from-scratch CNN can hit 95%+ here. The application is meaningful: phone-based crop disease detection for farmers in developing countries.

**HAM10000 / Skin Cancer** (10,015 dermatoscopic images, 7 lesion types including melanoma)
[Kaggle download](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000)

The class distribution is heavily imbalanced (most images are benign nevi). Accuracy alone is misleading here, since a model that always predicts "benign" scores 67% but is medically useless. Forces you to learn weighted loss functions, stratified sampling, and metrics like precision/recall.

**Garbage Classification** (~2,500 images, 6 recycling categories: glass, paper, plastic, metal, cardboard, trash)
[Kaggle download](https://www.kaggle.com/datasets/sumn2u/garbage-classification-v2)

Small dataset, intuitive classes, environmental application. Some items are genuinely ambiguous (is that cardboard or paper?), which makes the confusion matrix interesting. You could even test it with photos from your own desk.

## The Pipeline

Every image classification project follows roughly the same steps. The details change per dataset, but the structure doesn't. Work through these phases in order, since each one builds on the previous.

### Phase 1: Get and Understand Your Data

Download the dataset. Before writing any model code, understand what you're working with.

- **How is it organized?** Folder-per-class (`train/normal/`, `train/pneumonia/`)? CSV with labels? Something else? Every dataset is different here.
- **Visualize samples.** Display a grid of images with labels. Can *you* tell the classes apart? What visual features distinguish them?
- **Check the basics.** How many images? How many classes? Are classes balanced or imbalanced? Are images all the same dimensions, or variable?
- **Spot potential problems early.** Corrupted files, wrong labels, extremely small or large images, classes that look nearly identical.

This phase feels like busywork but saves you hours of debugging later. Most training failures trace back to data issues you could have caught here.

### Phase 2: Preprocessing and Augmentation

CNNs need consistent input dimensions and benefit enormously from augmentation. This is the biggest difference from MLP preprocessing.

**Resize to uniform dimensions.** All images in a batch must be the same size. If your dataset has variable dimensions, pick a target size (128x128 is a good default). Use `RandomResizedCrop` for training (augmentation + resize in one step) and `Resize` + `CenterCrop` for validation.

**Normalize per channel.** Use ImageNet values `mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)` unless you have a reason not to. For grayscale, use `mean=(0.5,), std=(0.5,)` or compute from your data.

**Design augmentation (training only).** This is where you effectively multiply your dataset size. Common augmentations:

| Augmentation | What it does | Use when |
|---|---|---|
| `RandomHorizontalFlip` | Mirror left-right | Almost always (not for text/digits) |
| `RandomResizedCrop` | Random crop + resize | Variable-size real photos |
| `RandomCrop` with padding | Pad edges, crop back | Small fixed-size images |
| `ColorJitter` | Brightness/contrast/saturation | When color matters for classification |
| `RandomRotation` | Rotate by random angle | When rotation is plausible |

**Important:** validation and test data is NEVER augmented. It needs to be deterministic so you can compare results consistently.

For small datasets (under 2,000 images), augmentation isn't optional. Without it your model will memorize the training set.

### Phase 3: Data Pipeline

Build the PyTorch machinery that feeds batches to your model.

**Create a Dataset.** If the data is organized as folder-per-class, `torchvision.datasets.ImageFolder(root, transform=...)` does everything automatically. Otherwise, write a custom `Dataset` with `__getitem__` that loads an image, applies transforms, and returns `(image_tensor, label)`.

**Split into train/val(/test).** Most datasets come with a split. If not, use `train_test_split` with `stratify` to keep class proportions balanced. Hold out 15-20% for validation.

**Create DataLoaders.** `batch_size=64`, `shuffle=True` for training, `num_workers=2`, `pin_memory=True` for GPU.

**Verify before continuing.** Grab one batch, check the shape is `(batch_size, channels, H, W)`, check dtypes (float32 images, long labels), display a few augmented samples to make sure transforms look reasonable.

### Phase 4: Design Your CNN

The architecture follows a simple pattern: stack conv blocks that halve spatial dimensions and double channels, until you reach 1x1 or 2x2, then classify.

**The conv block:** `Conv2d(stride=2, padding=ks//2)` > `BatchNorm2d` > `ReLU`. Stride-2 halves dimensions each layer. BatchNorm is not optional for networks deeper than 2-3 layers. The final layer has no BN and no activation (raw logits for `CrossEntropyLoss`).

**How deep?** Count how many times you need to halve to get from your image size to 2x2 or 1x1:

| Image size | Layers needed | Example channels |
|---|---|---|
| 32x32 | 5 (32>16>8>4>2>1) | 16>32>64>128>classes |
| 128x128 | 7 (128>64>32>16>8>4>2>1) | 16>32>64>128>256>512>classes |
| 224x224 | 7-8 with GAP | 16>32>64>128>256>512, then AdaptiveAvgPool2d(1) |

**First layer kernel:** Use 5x5 instead of 3x3 for the first conv layer. A 3x3 on 3 channels = 27 inputs going to 16 outputs (expanding, wasteful). 5x5 = 75 inputs to 16 (compressing, better).

**If spatial size doesn't reach exactly 1x1:** use `nn.AdaptiveAvgPool2d(1)` after the last conv block, then a `nn.Linear` for classification. This works for any input size.

**Verify:** pass a dummy batch through, confirm output shape is `(batch_size, num_classes)`. Count parameters.

### Phase 5: Train

**Setup:**
- Loss: `nn.CrossEntropyLoss()` (handles class imbalance with `weight` parameter if needed)
- Optimizer: `torch.optim.Adam(model.parameters(), lr=0.01)`
- Scheduler: `OneCycleLR(optimizer, max_lr=0.01, steps_per_epoch=len(train_loader), epochs=num_epochs)`
- Start with 25-30 epochs

**The loop:** For each epoch, run all training batches (forward > loss > backward > optimizer.step > scheduler.step), then validate without gradients. Track train_loss, val_loss, val_accuracy per epoch. Save the best model checkpoint based on val accuracy.

**Critical detail:** `OneCycleLR` steps every batch, not every epoch. Call `scheduler.step()` inside the batch loop, right after `optimizer.step()`.

**Plot training curves.** Loss curves (train + val), accuracy curve, LR schedule. A growing gap between train and val loss means overfitting. If train loss barely decreases, the learning rate is too low or the model is too small.

### Phase 6: Evaluate

Don't just look at the accuracy number. Understand *where* and *why* the model fails.

- **Confusion matrix.** Which classes get confused? Are the confusions sensible (e.g. two similar-looking defect types) or surprising?
- **Per-class accuracy.** Which classes are easiest/hardest? For imbalanced datasets, overall accuracy can be misleading.
- **Visualize predictions.** Show the most confident correct predictions and the worst failures. What do the failures have in common?
- **Learned filters.** Extract conv1 weights and display them as small images. You should see edge and color gradient detectors that the network discovered through training.

### Phase 7: Experiment and Improve

Once you have a baseline, start experimenting. The most impactful changes, in order of typical importance:

1. **Augmentation.** If your dataset is small, adding more augmentation types (ColorJitter, RandomRotation, RandomErasing) can help a lot. But too aggressive augmentation on tiny datasets can hurt because the model sees too many distorted views to learn anything.

2. **Model size.** More parameters isn't better with limited data. Try halving the channels. If accuracy stays the same or improves, your model was too big. Try doubling them. If it gets worse, you're overfitting.

3. **Depth.** If your deepest layers have mostly dead activations (90%+ near zero), the network is too deep for your data. Remove a layer or two.

4. **Learning rate.** Try `max_lr` values of 0.001, 0.003, 0.01, 0.03. The right value depends on your dataset and model size.

5. **MLP baseline.** Train a quick MLP on the same data. The gap between CNN and MLP accuracy tells you how much spatial structure matters for your task.

6. **Pixel shuffle test.** Scramble all pixels (same random permutation for every image). CNN should collapse, MLP shouldn't care. This proves your CNN is using spatial patterns, not just color statistics.

### What's Next

This project trains a CNN from random weights. The next step is **transfer learning**: take a model (ResNet, EfficientNet) already trained on millions of images, and fine-tune it for your specific task. You'll see accuracy jump from 40% to 90%+ because the pretrained model already knows edges, textures, shapes, and object parts. From-scratch training becomes obsolete for most real problems.

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>